Importar librerìas

In [4]:
import pandas as pd
import numpy as np

# 1. Cargar archivo original
# ==============================================================


In [5]:

ruta = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\df_completo_con_campo.xlsx"
df = pd.read_excel(ruta)

In [6]:
# Asegurar formato de fecha
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Ordenar por pozo y fecha
df = df.sort_values(["pozo", "date"]).reset_index(drop=True)

# 2. Calcular slope_7 y slope_14 (pendiente local)
# ==============================================================


In [10]:
def slope_window(y):
    x = np.arange(len(y))
    if len(y) < 2:
        return 0
    m, b = np.polyfit(x, y, 1)
    return m

df["slope_7"] = (
    df.groupby("pozo")["prueba_pozooil_24__prueba_pozowater_24"]
    .rolling(window=7, min_periods=3)
    .apply(slope_window)
    .reset_index(level=0, drop=True)
)

df["slope_14"] = (
    df.groupby("pozo")["prueba_pozooil_24__prueba_pozowater_24"]
    .rolling(window=14, min_periods=5)
    .apply(slope_window)
    .reset_index(level=0, drop=True)
)

# ==============================================================
# 3. Calcular delta_1 y delta_3 (saltos puntuales)




In [8]:
df["delta_1"] = df.groupby("pozo")["prueba_pozooil_24__prueba_pozowater_24"].diff()
df["delta_3"] = df.groupby("pozo")["prueba_pozooil_24__prueba_pozowater_24"].diff(3)

# 4. Calcular umbrales por pozo


In [11]:
# Umbrales para slope
stats_slope = df.groupby("pozo")["slope_7"].agg(["mean", "std"])
stats_slope["umbral_neg"] = stats_slope["mean"] - 2 * stats_slope["std"]
stats_slope["umbral_pos"] = stats_slope["mean"] + 2 * stats_slope["std"]

# Umbrales para delta (saltos)
stats_delta = df.groupby("pozo")["delta_1"].quantile([0.10, 0.90]).unstack()
stats_delta.columns = ["p10", "p90"]

# ==============================================================
# 5. Clasificador híbrido


In [12]:
def clasificar_evento(row):

    pozo = row["pozo"]

    slope_u_neg = stats_slope.loc[pozo, "umbral_neg"]
    slope_u_pos = stats_slope.loc[pozo, "umbral_pos"]

    delta_p10 = stats_delta.loc[pozo, "p10"]
    delta_p90 = stats_delta.loc[pozo, "p90"]

    # Regla híbrida
    if (row["slope_7"] < slope_u_neg) or (row["delta_1"] < delta_p10):
        return 1   # caída fuerte

    if (row["slope_7"] > slope_u_pos) or (row["delta_1"] > delta_p90):
        return 2   # reacondicionamiento

    return 0        # normal

df["evento_hibrido"] = df.apply(clasificar_evento, axis=1)

# ==============================================================

# 6. Guardar archivo final
# ==============================================================

In [13]:
ruta_salida = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\df_con_eventos_hibridos.xlsx"
df.to_excel(ruta_salida, index=False)

print("✔ Proceso completado.")
print("✔ Archivo guardado en:", ruta_salida)

✔ Proceso completado.
✔ Archivo guardado en: C:\Users\Vìctor\OneDrive\Desktop\ESP_Project\df_con_eventos_hibridos.xlsx
